# Atelier Colab GPU worker

This notebook runs the shipped `backend/scripts/colab_worker.py` on a real CUDA runtime. It is an ephemeral, local open-source worker, not a hosted generation service. Upload that file to the Colab session before starting the API.

In [ ]:
%pip install -q fastapi uvicorn[standard] python-multipart pillow pydantic transformers accelerate torchvision safetensors diffusers huggingface_hub

In [ ]:
import os, torch
if not torch.cuda.is_available():
    raise RuntimeError('NO_GPU: switch Colab to a CUDA runtime before starting the worker')
print(torch.cuda.get_device_name(0), torch.cuda.device_count())

In [ ]:
# Optional persistence. Keep source masks, identity profiles and outputs across sessions.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['WORKER_DATA_ROOT'] = '/content/drive/MyDrive/atelier-worker'
else:
    os.environ['WORKER_DATA_ROOT'] = '/content/atelier-worker'

# Use the exact shared secret. This is not the ngrok authentication token.
try:
    from google.colab import userdata
    os.environ['COLAB_WORKER_TOKEN'] = userdata.get('COLAB_WORKER_TOKEN')
except Exception:
    os.environ['COLAB_WORKER_TOKEN'] = os.environ.get('COLAB_WORKER_TOKEN', '')
os.environ['WORKER_PROVIDER'] = 'qwen-image-edit'
os.environ['ATELIER_MODEL_ROOT'] = '/content/atelier-models'
# The Qwen pipeline is already loaded in this notebook as `pipe`; it will
# be injected into the worker process below, so no model download is needed.
os.environ['WORKER_PROVIDER_COMMAND'] = 'in-process:qwen-image-edit'
if not os.environ['COLAB_WORKER_TOKEN']:
    raise RuntimeError('WORKER_AUTH_FAILED: configure COLAB_WORKER_TOKEN in Colab Secrets')

In [ ]:
# Upload the shipped worker. The provider uses the already-loaded notebook `pipe`.
from google.colab import files
uploaded = files.upload()
required_uploads = {'colab_worker.py'}
missing_uploads = required_uploads - set(uploaded)
if missing_uploads:
    raise RuntimeError(f'Upload the shipped worker and provider adapter: {sorted(missing_uploads)}')
print({'provider': os.environ['WORKER_PROVIDER'], 'using_existing_pipe': 'pipe' in globals()})


In [ ]:
# Confirm the existing Qwen pipeline is still alive. Do not download it again.
if 'pipe' not in globals():
    raise RuntimeError('QWEN_PIPELINE_NOT_FOUND: rerun the cell that loads QwenImageEditPipeline before continuing')
print({'pipeline': type(pipe).__name__, 'provider': os.environ['WORKER_PROVIDER']})

In [ ]:
# The existing in-memory Qwen pipeline is audited by the worker startup probe below.
print('In-process Qwen probe will run before the worker accepts requests.')


In [ ]:
# Configure the in-process provider around the already-loaded Qwen `pipe`.
if 'pipe' not in globals():
    raise RuntimeError('QWEN_PIPELINE_NOT_FOUND')
from pathlib import Path
from PIL import Image
import torch
class ExistingQwenProvider:
    name = 'qwen-image-edit'
    capabilities = ['image_edit', 'reference_edit', 'product_reference']
    def __init__(self, pipeline):
        self.pipeline = pipeline
        self._probe = None
    def startup(self):
        if self._probe is not None:
            return self._probe
        try:
            output = Path('/content/atelier-worker/startup-probe.png')
            output.parent.mkdir(parents=True, exist_ok=True)
            image = Image.new('RGB', (512, 512), (220, 210, 195))
            result = self.pipeline(image=[image], prompt='A studio product photograph of a handbag.', true_cfg_scale=4.0, num_inference_steps=4, width=512, height=512, generator=torch.Generator(device='cuda').manual_seed(1)).images[0]
            result.save(output, format='PNG')
            self._probe = {'status': 'online', 'configured': True, 'model_loaded': True, 'inference_passed': output.is_file(), 'provider': self.name, 'output': str(output)}
        except Exception as exc:
            self._probe = {'status': 'failed', 'configured': True, 'model_loaded': True, 'inference_passed': False, 'provider': self.name, 'reason': f'{type(exc).__name__}: {exc}'}
        return self._probe
    def ready(self):
        return self.startup().get('inference_passed') is True
    def generate(self, *, references, prompt, output, seed):
        if not self.ready():
            raise RuntimeError('PROVIDER_INFERENCE_FAILED: Qwen startup probe failed')
        images = [Image.open(path).convert('RGB') for path in references[:6]]
        result = self.pipeline(image=images, prompt=prompt, negative_prompt='text, watermark, extra product, redesigned product, deformed anatomy, duplicate product', true_cfg_scale=4.0, num_inference_steps=8, width=768, height=1024, generator=torch.Generator(device='cuda').manual_seed(seed)).images[0]
        result.save(output, format='PNG')
provider = ExistingQwenProvider(pipe)
provider_startup = provider.startup()
print(provider_startup)
if not provider.ready():
    raise RuntimeError('QWEN_PROVIDER_PROBE_FAILED: see provider_startup above')

In [ ]:
# Refuse to start the wrong FastAPI application.
import importlib.util
spec = importlib.util.spec_from_file_location('atelier_colab_worker', '/content/colab_worker.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
required = {'/health', '/verify', '/generate', '/job/{job_id}', '/batch', '/cancel/{job_id}'}
actual = {route.path for route in module.app.routes}
missing = required - actual
if missing:
    raise RuntimeError(f'INCOMPATIBLE_WORKER: missing routes {sorted(missing)}')
print({'worker': 'atelier-colab', 'required_routes_present': True, 'paths': sorted(required)})

In [ ]:
import threading, time, requests, uvicorn
# Run Uvicorn in this same Python process so it can access the injected `pipe`.
server = uvicorn.Server(uvicorn.Config(module.app, host='0.0.0.0', port=7860, log_level='info'))
worker_thread = threading.Thread(target=server.run, daemon=True)
worker_thread.start()
for _ in range(60):
    try:
        response = requests.get('http://127.0.0.1:7860/health', headers={'Authorization': f'Bearer {os.environ["COLAB_WORKER_TOKEN"]}'}, timeout=10)
        if response.ok:
            print(response.json())
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError('WORKER_START_FAILED: authenticated health check did not succeed')

In [ ]:
# Real operator verification. Upload 1-6 product references; this pass is ephemeral and must be rerun after restart.
verification_upload = files.upload()
references = []
import base64, mimetypes
for filename, data in verification_upload.items():
    references.append({'filename': filename, 'content_type': mimetypes.guess_type(filename)[0] or 'image/png', 'data_base64': base64.b64encode(data).decode('ascii')})
if not references:
    raise RuntimeError('VERIFICATION_REQUIRED: upload real product references')
verify_response = requests.post('http://127.0.0.1:7860/verify', headers={'Authorization': f'Bearer {os.environ["COLAB_WORKER_TOKEN"]}'}, json={'references': references, 'identity_profile': {'category': 'handbag'}, 'prompt': 'Verify a real human visibly carrying the exact uploaded product.'}, timeout=1900)
verify_response.raise_for_status()
print('verification:', verify_response.json())
print('final health:', requests.get('http://127.0.0.1:7860/health', headers={'Authorization': f'Bearer {os.environ["COLAB_WORKER_TOKEN"]}'}, timeout=30).json())
print('Configure the Replit worker URL and the same token without printing either secret. Verification is ephemeral and must be rerun after every Colab runtime restart.')